In [1]:
import os
import math
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pickle
%matplotlib inline
from IPython.display import clear_output
from typing import List, Tuple, Optional

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

# data_folder = "drive/MyDrive/mestrado_final_files/data/"
# models_folder = "drive/MyDrive/mestrado_final_files/models/NGramModel_ExtraInfo/"

data_folder = "G:\Meu Drive\mestrado_final_files\data\\"
final_simulations_folder = "G:\Meu Drive\mestrado_final_files\simulations\\FinalSimulations\\"
models_token_folder = "G:\Meu Drive\mestrado_final_files\models\\NGramModel_ExtraInfo\\"
models_time_folder = "G:\Meu Drive\mestrado_final_files\models\\SpentTimeModel\\"


In [2]:
print(f"Pandas version: {pd.__version__},\nNumpy Version: {np.__version__},\
      \nTorch Version: {torch.__version__},\nMatplotlib Version: {plt.matplotlib.__version__}")

Pandas version: 2.2.3,
Numpy Version: 2.2.6,      
Torch Version: 2.8.0+cpu,
Matplotlib Version: 3.10.0


In [3]:
%run 02_functions_simulations.ipynb

In [4]:
tokens_info = pd.read_csv(data_folder + 'tokens_info.txt', sep = ';')\
.sort_values(['cod', 'token'])\
.set_index('token')

# Convert the index of tokens_info to a list of tokens
list_tokens = tokens_info.index.to_list()

# Create a dictionary mapping each token to a unique integer, starting from 1
token_to_integer = {tok: idx + 1 for idx, tok in enumerate(list_tokens)}

# Assign a special integer (0) for the period (.)
token_to_integer['.'] = 0

# Create an inverse mapping from integers back to tokens
integer_to_token = {idx: tok for tok, idx in token_to_integer.items()}

# tokens_info = tokens_info\
# .query("hpts != 0 or vpts != 0 or limhteam != 0 or limvteam != 0")\
# .to_dict(orient = 'index')


In [5]:
with open(data_folder + 'dict_set_covariables.pkl', 'rb') as f:
    dict_set_covariables = pickle.load(f)

In [6]:
with open(data_folder + 'prop_end_period_by_remaining_time.pkl', 'rb') as f:
    dict_prop_end_period_remaining_time = pickle.load(f)

In [7]:
list_models = [item for item in os.listdir(models_token_folder) if "NGramModel_ExtraInfo_seasondata" in item and ".pth" in item]
list_models = [item for item in list_models if "_context5_embedding32_numlayers3_numneurons50_nodropout_layernormFalse_setcovariables2_parameters" in item]

In [8]:
VOCAB_SIZE = 601
TIME_SIZE = 301

In [9]:
list_models

['NGramModel_ExtraInfo_seasondata201819_context5_embedding32_numlayers3_numneurons50_nodropout_layernormFalse_setcovariables2_parameters.pth',
 'NGramModel_ExtraInfo_seasondata201920_context5_embedding32_numlayers3_numneurons50_nodropout_layernormFalse_setcovariables2_parameters.pth',
 'NGramModel_ExtraInfo_seasondata202021_context5_embedding32_numlayers3_numneurons50_nodropout_layernormFalse_setcovariables2_parameters.pth',
 'NGramModel_ExtraInfo_seasondata202122_context5_embedding32_numlayers3_numneurons50_nodropout_layernormFalse_setcovariables2_parameters.pth',
 'NGramModel_ExtraInfo_seasondata202223_context5_embedding32_numlayers3_numneurons50_nodropout_layernormFalse_setcovariables2_parameters.pth']

In [10]:
i = 0
for model_filename in list_models:
    i += 1
    print(i)

    time_model_filename = model_filename.replace("NGramModel_ExtraInfo", "SpentTimeModel")

    season_ = model_filename.split("_seasondata")[1].split("_")[0]
    CONTEXT_SIZE = int(model_filename.split("_context")[1].split("_")[0])
    EMBEDDING_LAYER_SIZE = int(model_filename.split("_embedding")[1].split("_")[0])
    variables_config = int(model_filename.split("_setcovariables")[1].split("_")[0])
    numerical_input_size = len(dict_set_covariables['index'][variables_config])
    num_hidden_neurons = int(model_filename.split("_numneurons")[1].split("_")[0])
    num_layers = int(model_filename.split("_numlayers")[1].split("_")[0])
    dropout_rate = 0 if "nodropout" in model_filename else int(model_filename.split("_dropout")[1].split("_")[0]) / 10
    layer_norm = True if model_filename.split("_layernorm")[1].split("_")[0] == 'True' else False

    token_model = NGramModel_ExtraInfo(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, embedding_dim = EMBEDDING_LAYER_SIZE, 
                                       numerical_input_dim = numerical_input_size, num_hidden_neurons = num_hidden_neurons, 
                                       num_layers = num_layers, activation = nn.GELU, dropout_rate = dropout_rate, 
                                       use_layer_norm = layer_norm)

    token_model.load_state_dict(torch.load(models_token_folder + model_filename, weights_only = True, map_location = torch.device('cpu')))
    token_model.eval()

    time_model = SpentTimeModel(context_size = CONTEXT_SIZE, vocab_size = VOCAB_SIZE, time_size = TIME_SIZE, 
                                embedding_dim = EMBEDDING_LAYER_SIZE, numerical_input_dim = numerical_input_size,
                                num_hidden_neurons = num_hidden_neurons, num_layers = num_layers,
                                activation_function = nn.GELU, dropout_rate = dropout_rate, use_layer_norm = layer_norm)


    time_model.load_state_dict(torch.load(models_time_folder + time_model_filename, weights_only = True, map_location = torch.device('cpu')))
    time_model.eval()


    for slot in range(0, 10):

        parquet_filename = ("simulations_" + model_filename.replace("_parameters.pth", "") + "_parte" + str(slot + 1) + ".parquet")
        # df_simulations = generate_game_plays(token_model = token_model, time_model = time_model, num_simulations = 10000, 
        #                                     token_context = CONTEXT_SIZE, num_numerical_variables = numerical_input_size, 
        #                                     config = variables_config)
        
        df_simulations = generate_game_plays_v2(class_model = "ExtraInfo", token_model1 = token_model, time_model1 = time_model, 
                                                num_simulations = 10000, token_context = CONTEXT_SIZE, num_numerical_variables = numerical_input_size,
                                                config_setvariables = variables_config, matrix_end_period_probs = dict_prop_end_period_remaining_time[season_])

        df_simulations.to_parquet(final_simulations_folder + parquet_filename, index = False, compression = "zstd")
        clear_output(wait = True)



Period: 7
keep_generating vector distribution: {0: 10000}
                  count      mean        std   min    1%    5%    10%    25%  \
home_points       10000  115.2574  11.451399  69.0  89.0  97.0  101.0  108.0   
away_points       10000  113.5259  11.053159  75.0  88.0  96.0   99.0  106.0   
perc_played_time  10000    0.0100   0.999950   0.0   0.0   0.0    0.0    0.0   

                    50%    75%    90%    95%    99%    max  
home_points       115.0  123.0  130.0  134.0  143.0  165.0  
away_points       113.0  121.0  128.0  132.0  140.0  164.0  
perc_played_time    0.0    0.0    0.0    0.0    0.0  100.0  
Result: {np.str_('away'): 4470, np.str_('home'): 5530}
